In [12]:
# Task 1.1

from pymongo import MongoClient
import json
from pprint import pprint

client = MongoClient("127.0.0.1", 27017)

db = client["DHS"]
if "IT_Devices" in db.list_collection_names():
    print("Dropping collection...")
    db.drop_collection("IT_Devices")
coll = db["IT_Devices"]

json_file = open("IT_Devices.json", 'r')
json_data = json.load(json_file)
coll.insert_many(json_data)

for item in coll.find():
    pprint(item)

Dropping collection...
{'Brand': 'HP',
 'Cost': 1099,
 'Date_Of_Purchase': '2022-02-10',
 'Screen_Size': 14,
 'Serial_Number': 'LAP-001',
 'Type': 'Laptop',
 'Weight': 1.35,
 '_id': ObjectId('68e37b3d48c7d00afba5b997')}
{'Brand': 'Lenovo',
 'Cost': 1599,
 'Date_Of_Purchase': '2023-09-18',
 'Screen_Size': 16,
 'Serial_Number': 'LAP-002',
 'Type': 'Laptop',
 'Weight': 2.45,
 '_id': ObjectId('68e37b3d48c7d00afba5b998')}
{'Brand': 'Apple',
 'Cost': 1899,
 'Date_Of_Purchase': '2024-05-25',
 'Screen_Size': 14,
 'Serial_Number': 'LAP-003',
 'Type': 'Laptop',
 'Weight': 1.25,
 '_id': ObjectId('68e37b3d48c7d00afba5b999')}
{'Brand': 'Dell',
 'Cost': 1249,
 'Date_Of_Purchase': '2022-01-08',
 'Screen_Size': 14,
 'Serial_Number': 'LAP-004',
 'Type': 'Laptop',
 'Weight': 1.55,
 '_id': ObjectId('68e37b3d48c7d00afba5b99a')}
{'Brand': 'ASUS',
 'Cost': 2199,
 'Date_Of_Purchase': '2023-12-03',
 'Screen_Size': 17,
 'Serial_Number': 'LAP-005',
 'Type': 'Laptop',
 'Weight': 2.6,
 '_id': ObjectId('68e37b3d48

In [14]:
# Task 1.2

update = {"$set": {"Brand": "Hewlett-Packard"}}
update_query = {"Brand": {"$eq": "HP"}}

result = coll.update_many(update_query, update)


query = {"$or": [
    {"$and": [
        {"Type": "Laptop"},
        {"Weight": {"$lt": 2}}
    ]},
    {"$and": [
        {"Type": "Tablet"},
        {"Battery_Capacity": {"$gte": 8000}}
    ]}
]}


for item in coll.find(query):
    if item["Type"] == "Laptop":
        print(f"Serial No: {item["Serial_Number"]}, Type: {item["Type"]}, Brand: {item["Brand"]}, Weight: {item["Weight"]}")
    elif item["Type"] == "Tablet":
        print(f"Serial No: {item["Serial_Number"]}, Type: {item["Type"]}, Brand: {item["Brand"]}, Battery Capacity: {item["Battery_Capacity"]}")

Serial No: LAP-001, Type: Laptop, Brand: Hewlett-Packard, Weight: 1.35
Serial No: LAP-003, Type: Laptop, Brand: Apple, Weight: 1.25
Serial No: LAP-004, Type: Laptop, Brand: Dell, Weight: 1.55
Serial No: LAP-006, Type: Laptop, Brand: Acer, Weight: 1.45
Serial No: TAB-001, Type: Tablet, Brand: Samsung, Battery Capacity: 8000
Serial No: TAB-002, Type: Tablet, Brand: Apple, Battery Capacity: 9720


In [28]:
# Task 1.3

import sqlite3

conn = sqlite3.connect("DHS.db")
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS Device(
    Serial_Number TEXT PRIMARY KEY,
    Type TEXT NOT NULL,
    Brand TEXT NOT NULL,
    Cost INTEGER NOT NULL,
    Date_Of_Purchase TEXT NOT NULL
)
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS Laptop(
    Serial_Number TEXT PRIMARY KEY REFERENCES Device(Serial_Number),
    Weight FLOAT NOT NULL,
    Screen_Size INTEGER NOT NULL
)
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS Tablet(
    Serial_Number TEXT PRIMARY KEY REFERENCES Device(Serial_Number),
    Battery_Capacity INTEGER NOT NULL
)
""")

conn.commit()

devices = list(coll.find())
laptops = list(coll.find({"Type": {"$eq": "Laptop"}}))
tablets = list(coll.find({"Type": {"$eq": "Tablet"}}))

devices_list = []
laptops_list = []
tablets_list = []

for d in devices:
    devices_list.append([d["Serial_Number"], d["Type"], d["Brand"], d["Cost"], d["Date_Of_Purchase"]])

for l in laptops:
    laptops_list.append([l["Serial_Number"], l["Weight"], l["Screen_Size"]])

for t in tablets:
    tablets_list.append([t["Serial_Number"], t["Battery_Capacity"]])

cur.executemany("""
INSERT INTO Device(Serial_Number, Type, Brand, Cost, Date_Of_Purchase) VALUES (?, ?, ?, ?, ?)
""", devices_list)

cur.executemany("""
INSERT INTO Laptop(Serial_Number, Weight, Screen_Size) VALUES (?, ?, ?)
""", laptops_list)

cur.executemany("""
INSERT INTO Tablet(Serial_Number, Battery_Capacity) VALUES (?, ?)
""", tablets_list)

conn.commit()
conn.close()

In [49]:
# Task 1.4

import datetime

conn = sqlite3.connect("DHS.db")
cur = conn.cursor()

cur.execute("SELECT Serial_Number, Type, Date_Of_Purchase FROM Device")

devices = cur.fetchall()

old_devices = []

now_date = datetime.datetime.now()

for device in devices:
    date = device[2]
    y = int(date[:4])
    m = int(date[5:7])
    d = int(date[8:10])

    device_date = datetime.datetime(y, m, d)
    
    device_age = (now_date - device_date).days 

    if device_age > 1000:
        old_devices.append(list(device) + [str(device_age)])

write_data = "Serial_Number,Type,Date_Of_Purchase,age of device(in days)\n"

for device in old_devices:
    write_data += ','.join(device) + '\n'

csv_file = open("Old_devices.csv", 'w')
csv_file.write(write_data.strip())
csv_file.close()